In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
from mscthesis.core.io import load_dataframe, save_dataframe 
from stillib_random import from_seed

df_dh = load_dataframe("./raw/dehydrated_dataset_Momayyezi_et_al_2022.csv")
df_w = load_dataframe("./raw/watered_dataset_Momayyezi_et_al_2022.csv")
root = from_seed(123456)

for df, condition in zip([df_dh, df_w], ["dehydrated", "watered"], strict=True):
    df["species"] = "Juglans Regia"
    df["PFT"] = "deciduous angiosperms"
    df["condition"] = condition
    df["An"] = df["assimilation_rate"]
    df["An error"] = df["d_assimilation_rate"]
    df["gs"] = df["stomatal_conductance"]
    df["gs error"] = df["d_stomatal_conductance"]
    df["gias"] = df["ias_conductance"]
    df["gias error"] = df["d_ias_conductance"]
    df["Ca"] = df["atmospheric_CO2"]
    df["Cistar"] = df["compensation_point"]

df = pd.concat([df_dh, df_w], ignore_index=True)
df = df[["species", "PFT", "accession", "condition", "An", "An error", "gs", "gs error", "gias", "gias error", "Ca", "Cistar"]]
df["Ci"] = np.nan 
df["Ci error"] = np.nan 

n_samples = 500

def draw_values(rng: np.random.Generator, mean: float, std: float, n: int, positive: bool = True) -> np.ndarray:
    values = np.zeros(n)
    index  = 0
    while index < n:
        sample = rng.normal(loc=mean, scale=std)
        if not positive or sample > 0:
            values[index] = sample
            index += 1
    return values


for i, row in df.iterrows():
    # draw random An and gs values from normal distributions
    An_values = draw_values(root.spawn("An").generator(), row["An"], row["An error"], n_samples)
    gs_values = draw_values(root.spawn("gs").generator(), row["gs"], row["gs error"], n_samples)
    # calculate Ci values
    Ci_values = row["Ca"] - (An_values / gs_values)
    # assign mean of Ci values to the dataframe
    df.at[i, "Ci"] = np.median(Ci_values)
    df.at[i, "Ci error"] = np.std(Ci_values)

save_dataframe("./derived/Momayyezi_filtered.csv", df)
    
